# sentiment-kg-build
## Preprocessing Pipeline for Sentiment Knowledge Graph

This notebook prepares two types of data for the sentiment knowledge graph:

1. **Sentiment data** — extracts aspect candidates from graduate descriptions and quotes using spaCy, then scores each aspect's sentiment using Aspect-Based Sentiment Analysis (ABSA)
2. **Network data** — cleans and encodes graduate activity and company affiliations into weighted affiliation matrices

**Inputs:** `text_sword80.csv`, `net_sword80.csv`  
**Outputs:** `text_aspec_senti_desc.csv`, `text_aspec_senti_quotes.csv`, `rolescomps_num.csv`

## Environment Setup

Requires the conda environment from the [Aspect-Based Sentiment Analysis](https://github.com/ScalaConsultants/Aspect-Based-Sentiment-Analysis) repo.

```bash
git clone https://github.com/ScalaConsultants/Aspect-Based-Sentiment-Analysis.git
cd Aspect-Based-Sentiment-Analysis
conda env create -f environment.yml
conda activate Aspect-Based-Sentiment-Analysis
pip install -e .
python -m spacy download en_core_web_sm
```

> Set the notebook kernel to the `Aspect-Based-Sentiment-Analysis` conda environment (Python 3.7.16).

In [ ]:
import pandas as pd
import spacy
import re
import ast
import time
from collections import defaultdict

import aspect_based_sentiment_analysis as absa
from transformers import BertTokenizer

---
## Part 1: Sentiment Data

### 1.1 Split Text into Descriptions and Quotes

Graduate text comes in two forms:
- **Description** — written by peers; captures *perceived* sentiment
- **Quote** — chosen by the graduate; captures *self-expressed* sentiment

Each is extracted into its own CSV for separate processing.

In [ ]:
input_file = 'processed_dat/text_dat/text_sword80.csv'
df = pd.read_csv(input_file)

# Description (perceived sentiment)
df[['id', 'Description']].to_csv('processed_dat/text_dat/text_descrips.csv', index=False)

# Quote (self-expressed sentiment)
df[['id', 'Quote']].to_csv('processed_dat/text_dat/text_quotes.csv', index=False)

print("Saved: text_descrips.csv, text_quotes.csv")

### 1.2 Load ABSA Pipeline

Loads the BERT-based classifier (`absa/classifier-rest-0.2`) and builds the inference pipeline manually, as the default pipeline call had compatibility issues with this environment.

Pipeline steps:
1. `preprocess` — creates a task object with one Example per aspect
2. `tokenize` — breaks each Example into tokens
3. `encode` — converts tokens into tensors for BERT input
4. `predict` — runs inference; returns sentiment scores and predicted classes (positive / negative / neutral)
5. `review` — links predictions back to original Examples
6. `postprocess` — assembles the final completed task

In [ ]:
name = 'absa/classifier-rest-0.2'

model        = absa.BertABSClassifier.from_pretrained(name)
tokenizer    = BertTokenizer.from_pretrained(name)
professor    = absa.Professor()
text_splitter = absa.sentencizer()  # requires en_core_web_sm

nlp = absa.Pipeline(model, tokenizer, professor, text_splitter)
print("ABSA pipeline loaded.")

#### Pipeline sanity check

Run the pipeline on a test sentence to confirm it is working before processing the full dataset.

In [ ]:
test_text    = "The food was great, but the service was awful."
test_aspects = ["food", "service"]

task             = nlp.preprocess(text=test_text, aspects=test_aspects)
tokenized        = nlp.tokenize(task.examples)
encoded          = nlp.encode(tokenized)
output           = nlp.predict(encoded)
predictions      = nlp.review(tokenized, output)
completed_task   = nlp.postprocess(task, predictions)

for result in completed_task.examples:
    print(f"Aspect: {result.aspect:12s}  Sentiment: {result.sentiment.name}")

### 1.3 Extract Aspect Candidates with spaCy

Aspects are extracted using spaCy dependency parsing. The approach follows:
- Hu & Liu (2004) for noun-based aspect extraction
- [Joshi (2018)](https://achyutjoshi.github.io/aspect_extraction/overview) for dependency path filtering
- [SetFit ABSA (2023)](https://huggingface.co/blog/setfit-absa) for noun compound handling

Three extraction strategies are combined:
1. **Noun chunks** filtered by syntactic dependency role (subject, object, etc.)
2. **Fallback standalone nouns** not captured by chunks
3. **Verb lemmas** from progressive, perfect, modal, and complement constructions — included because graduate profiles use action-oriented language

A custom Filipino/English stopword list handles noise from bilingual text.

In [ ]:
nlp_spacy = spacy.load("en_core_web_sm")

def clean_text(text):
    """Normalize punctuation and remove quote attributions (e.g., '— Ralph Waldo Emerson')."""
    text = re.sub(r'[-–—]\s*[A-Z][\w\.\s]+$', '', text).strip()
    text = re.sub(r'([.!?\u2026]+)(\w)', r'\1 \2', text)
    text = re.sub(r'[\.!\?\u2026]+', '.', text)
    return text

def deduplicate_phrases(aspects):
    """Remove redundant shorter phrases already covered by a longer one."""
    result = []
    seen_words = set()
    for phrase in sorted(aspects, key=lambda x: len(x.split())):
        words = set(phrase.split())
        if not words & seen_words:
            result.append(phrase)
            seen_words.update(words)
    return result

def extract_aspects(text):
    """
    Extract aspect candidates from a graduate text using spaCy.

    Returns a list of normalized strings (nouns, noun phrases, verb lemmas).
    Returns an empty list for missing or very short text.
    """
    if pd.isna(text) or not str(text).strip():
        return []

    text = clean_text(text)
    text = re.sub(r'\"\s*[a-zA-Z]\s*\"', '', text)  # remove quoted single letters like "E"
    text = re.sub(r"'\s*[a-zA-Z]\s*'", '', text)

    if len(text.split()) <= 2:
        return [text.lower().strip()]

    doc = nlp_spacy(text)
    aspects = set()
    seen_phrases = set()

    # Stopwords: English function words + Filipino particles common in the corpus
    custom_exclude = {
        "sa", "din", "in", "on", "at", "is", "to", "it", "i", "me", "we",
        "he", "she", "you", "a", "an", "the", "mga", "i'll", "'s", "nang",
        "lang", "kayo", "ako", "ka", "budo", "are", "que", "ng", "mo", "ang",
        "iyong", "cayu", "cu", "ngan", "apo", "has", "have", "thing", "way",
        "do", "be", "there", "that", "this", "what", "which", "who", "how", "'"
    } | set("abcdefghijklmnopqrstuvwxyz")

    # 1. Noun chunks filtered by dependency role
    allowed_deps = {"nsubj", "dobj", "pobj", "attr", "nsubjpass", "appos"}
    for chunk in doc.noun_chunks:
        head = chunk.root
        if (
            head.pos_ not in {"PRON", "DET"}
            and head.dep_ in allowed_deps
            and len(chunk.text.strip()) > 1
        ):
            cleaned = " ".join(t.text for t in chunk if t.pos_ != "DET")
            normalized = cleaned.lower().strip()
            if normalized not in seen_phrases and normalized not in custom_exclude:
                aspects.add(normalized)
                seen_phrases.add(normalized)

    # 2. Fallback: standalone nouns not captured in any chunk
    for token in doc:
        if token.pos_ in {"NOUN", "PROPN"} and token.is_alpha and len(token.text) > 1:
            word = token.text.lower().strip()
            if word not in aspects and word not in custom_exclude:
                aspects.add(word)

    # 3. Verb lemmas from progressive, perfect, modal, and complement constructions
    for token in doc:
        aux_lemmas = {aux.lemma_ for aux in token.head.lefts}

        if token.tag_ == "VBG" and ("be" in aux_lemmas or "have" in aux_lemmas):
            lemma = token.lemma_.lower().strip()
            if lemma not in aspects and lemma not in custom_exclude:
                aspects.add(lemma)
        elif token.tag_ == "VBN" and "have" in aux_lemmas:
            lemma = token.lemma_.lower().strip()
            if lemma not in aspects and lemma not in custom_exclude:
                aspects.add(lemma)
        elif token.pos_ == "VERB" and token.head.tag_ == "MD":
            lemma = token.lemma_.lower().strip()
            if lemma not in aspects and lemma not in custom_exclude:
                aspects.add(lemma)
        elif token.pos_ == "VERB" and token.dep_ in {"ROOT", "conj"} and token.head.pos_ != "AUX":
            lemma = token.lemma_.lower().strip()
            if lemma not in aspects and lemma not in custom_exclude:
                aspects.add(lemma)
        elif token.tag_ == "VBG" and token.dep_ == "xcomp":
            lemma = token.lemma_.lower().strip()
            if lemma not in aspects and lemma not in custom_exclude:
                aspects.add(lemma)

    return deduplicate_phrases(aspects)

In [ ]:
# Apply aspect extraction to both text types
for text_col, in_file, out_file in [
    ("Description", "text_descrips.csv", "text_aspec_desc.csv"),
    ("Quote",       "text_quotes.csv",   "text_aspec_quotes.csv"),
]:
    df = pd.read_csv(f'processed_dat/text_dat/{in_file}')
    df['Aspects'] = df[text_col].apply(extract_aspects)
    df.to_csv(f'processed_dat/text_dat/{out_file}', index=False)
    print(f"Saved: {out_file}")

### 1.4 Score Aspect Sentiments with ABSA

Each aspect extracted in the previous step is scored using the BERT-based ABSA classifier.

**Quotes** are short enough to run row-by-row. **Descriptions** are longer and can exhaust CPU memory, so they are processed in batches of 16 rows with a small delay between batches to prevent memory spikes.

In [ ]:
def parse_aspects(aspect_str):
    """Parse the stringified list stored in the 'Aspects' column."""
    try:
        return ast.literal_eval(str(aspect_str))
    except Exception:
        return []

def score_sentiments(row, text_col):
    """
    Run ABSA inference for one row.
    Returns a dict mapping each aspect to its sentiment label (positive/negative/neutral).
    """
    text    = row[text_col]
    aspects = parse_aspects(row["Aspects"])
    if not text or not aspects:
        return {}
    try:
        task           = nlp.preprocess(text=text, aspects=aspects)
        tokenized      = nlp.tokenize(task.examples)
        encoded        = nlp.encode(tokenized)
        output         = nlp.predict(encoded)
        predictions    = nlp.review(tokenized, output)
        completed_task = nlp.postprocess(task, predictions)
        return {ex.aspect: ex.sentiment.name for ex in completed_task.examples}
    except Exception as e:
        print(f"Error on row {row.name}: {e}")
        return {}

In [ ]:
# Quotes — short texts, process row-by-row
df = pd.read_csv("processed_dat/text_dat/text_aspec_quotes.csv")
df["Sentiments"] = df.apply(lambda row: score_sentiments(row, "Quote"), axis=1)
df.to_csv("processed_dat/text_dat/text_aspec_senti_quotes.csv", index=False)
print("Saved: text_aspec_senti_quotes.csv")

In [ ]:
# Descriptions — longer texts, process in batches to manage memory
df         = pd.read_csv("processed_dat/text_dat/text_aspec_desc.csv")
batch_size = 16
batches    = []

for i in range(0, len(df), batch_size):
    print(f"Processing rows {i}–{min(i + batch_size, len(df))}...")
    batch = df.iloc[i:i + batch_size].copy()
    batch["Sentiments"] = batch.apply(lambda row: score_sentiments(row, "Description"), axis=1)
    batches.append(batch)
    time.sleep(0.2)  # brief pause to avoid memory spikes

df_result = pd.concat(batches)
df_result.to_csv("processed_dat/text_dat/text_aspec_senti_desc.csv", index=False)
print(f"Saved: text_aspec_senti_desc.csv  ({len(df_result)} rows)")

---
## Part 2: Network Data

Graduate social network data is encoded as an affiliation matrix — a two-mode structure linking each cadet (node) to the groups and activities they belonged to.

Two matrices are constructed:
- **Roles matrix** — cadet × activity affiliations, weighted by leadership level
- **Company matrix** — cadet × company affiliations, encoded as binary dummies

These are merged into a single weighted affiliation matrix for network analysis.

### 2.1 Clean Column Names

In [ ]:
df = pd.read_csv('processed_dat/net_sword80.csv')

df.columns = (
    df.columns
    .str.replace('""', '"')
    .str.replace('"', '')
    .str.strip()
    .str.replace(r'\s+', '_', regex=True)
    .str.rstrip('_')
    .str.replace(r'_+', '_', regex=True)
    .str.lower()
)

# Remove duplicate columns and fill missing values
df = df.loc[:, ~df.columns.duplicated()]
df = df.fillna(0).replace('', 0)

df.to_csv('processed_dat/net_dat/total_net_clean.csv', index=False)
print(f"Cleaned dataset: {df.shape[0]} rows, {df.shape[1]} columns")

### 2.2 Build Roles Matrix

Extract activity columns and encode each cadet's role using an ordinal scale:

| Level | Roles |
|---|---|
| 0 | No participation |
| 1 | General member |
| 2 | Operational staff (board, editorial) |
| 3 | Assistants and specialist roles |
| 4 | Mid-level editors and managers |
| 5 | Senior leadership and directors |
| 6 | Top executive leadership (president, cadet-in-charge) |

In [ ]:
df = pd.read_csv('processed_dat/net_dat/total_net_clean.csv')

# Subset to activity columns only
activity_cols = ['id'] + [col for col in df.columns if col.startswith('activities.')]
df_roles = df[activity_cols]
df_roles.to_csv('processed_dat/net_dat/net_roles.csv', index=False)
print(f"Roles matrix: {df_roles.shape[1] - 1} activity columns")

In [ ]:
role_map = {
    # 1 — General Members
    "Member": 1, "Participant": 1,

    # 2 — Operational Staff
    "Arts Staffer": 2, "Board and Staff": 2, "Editorial Board": 2,
    "Editorial Treasury Board": 2, "Ex-officio Member": 2,

    # 3 — Assistants and Specialist Roles
    "Assistant Coach": 3, "Underclass Assistant": 3, "Chairman on Decorations": 3,
    "Coordinator": 3, "Delegate": 3, "Representative": 3,
    "PMA Representative": 3, "Commentator": 3, "Basic Airborne Course Graduate": 3,

    # 4 — Mid-Level Editors and Managers
    "Associate Editor": 4, "Editor": 4, "The Editor": 4, "News Editor": 4,
    "Sports Editor": 4, "Features Editor": 4, "Photo Editor": 4,
    "Photography Editor": 4, "Section Editor": 4, "Sports Section Editor": 4,
    "Art Editor": 4, "Arts Editor": 4, "Literary Editor": 4, "Pilipino Editor": 4,
    "Administration Editor": 4, "Copyright Editor": 4, "Circulation Editor": 4,
    "Research Editor": 4, "Manager": 4, "Equipment Manager": 4,
    "Historian": 4, "Class Historian": 4, "Business Manager": 4,

    # 5 — Senior Leadership and Directors
    "Director": 5, "General Coordinator": 5, "Captain": 5, "Coach": 5,
    "Athletic Representative": 5, "Stage Director": 5, "Lights and Sounds Director": 5,
    "Tournament Director": 5, "Military Review Editor": 5,
    "Assistant Cadet-in-Charge": 5, "Managing Editor": 5,

    # 6 — Top Executive Leadership
    "President": 6, "Vice-President": 6, "Vice-President (Finance)": 6,
    "Governor General": 6, "Chairman": 6, "Editor-in-Chief": 6,
    "Chief Technical Director": 6, "Secretary-Treasurer": 6,
    "Secretary": 6, "Treasurer": 6, "Cadet-in-Charge": 6,
}

df = pd.read_csv('processed_dat/net_dat/net_roles.csv')

for col in df.columns:
    if col != 'id':
        df[col] = df[col].map(role_map).fillna(0).astype(int)

df.to_csv('processed_dat/net_dat/net_roles_num.csv', index=False)
print("Saved: net_roles_num.csv")

#### Consolidate activity column variants

Many columns represent the same activity under different names (e.g., company-specific team columns like `activities.d_co_basketball_team` are all the same activity as `activities.basketball_team`). These are remapped to canonical names and merged by summing.

In [ ]:
activity_map = {
    'activities.a_co_cross_country_team': 'activities.cross_country_team',
    'activities.a_co_gymnastics_team': 'activities.gymnastics_team',
    'activities.a_company': 'activities.a_company',
    'activities.b_basketball_team': 'activities.basketball_team',
    'activities.b_company': 'activities.b_company',
    'activities.c_co_basketball_team': 'activities.basketball_team',
    'activities.c_co_cross_country_team': 'activities.cross_country_team',
    'activities.c_co_judo_team_(jc)': 'activities.judo_team',
    'activities.d_co_basketball_team': 'activities.basketball_team',
    'activities.d_co_fencing_team': 'activities.fencing_team',
    'activities.d_co_karate_team': 'activities.karate_team',
    'activities.d_co_pelota_team': 'activities.pelota_team',
    'activities.d_co_soccer_team': 'activities.soccer_team',
    'activities.d_co_swimming_team': 'activities.swimming_team',
    'activities.d_co_track_team': 'activities.track_team',
    'activities.d_co_volleyball_team': 'activities.volleyball_team',
    'activities.d_co_wrestling_team': 'activities.wrestling_team',
    'activities.d_company': 'activities.d_company',
    'activities.e_co_archery_team': 'activities.archery_team',
    'activities.e_co_basketball_team': 'activities.basketball_team',
    'activities.e_co_boxing_team': 'activities.boxing_team',
    'activities.e_co_fencing_team': 'activities.fencing_team',
    'activities.e_co_gymnastics_team': 'activities.gymnastics_team',
    'activities.e_co_karate_team': 'activities.karate_team',
    'activities.e_co_lawn_tennis_team': 'activities.lawn_tennis_team',
    'activities.e_co_pelota_team': 'activities.pelota_team',
    'activities.e_co_soccer_team': 'activities.soccer_team',
    'activities.e_co_swimming_team': 'activities.swimming_team',
    'activities.e_co_track_team': 'activities.track_team',
    'activities.e_company': 'activities.e_company',
    'activities.en_co_volleyball_team': 'activities.volleyball_team',
    'activities.f_co_basketball_team': 'activities.basketball_team',
    'activities.f_co_fencing_team': 'activities.fencing_team',
    'activities.f_co_karate_team': 'activities.karate_team',
    'activities.f_co_pelota_team': 'activities.pelota_team',
    'activities.f_company': 'activities.f_company',
    'activities.g_co_boxing_team': 'activities.boxing_team',
    'activities.g_co_chess_team': 'activities.chess_team',
    'activities.g_co_fencing_team': 'activities.fencing_team',
    'activities.g_co_swimming_team': 'activities.swimming_team',
    'activities.g_co_wrestling_team': 'activities.wrestling_team',
    'activities.h_co_gymnastics_team': 'activities.gymnastics_team',
    'activities.h_co_softball_team': 'activities.softball_team',
    'activities.h_co_track_and_field_team': 'activities.track_and_field_team',
    'activities.h_co_volleyball_team': 'activities.volleyball_team',
    'activities.h_co_wrestling_team': 'activities.wrestling_team',
    'activities.h_company': 'activities.h_company',
    'activities.h_karate_team': 'activities.karate_team',
    'activities.activities.2nd_battalion,_choral_group': 'activities.choral_group',
    'activities.archery_club': 'activities.archery_club',
    'activities.archery_corps_squad': 'activities.archery_team',
    'activities.arnis_club': 'activities.arnis_club',
    'activities.arnis_corps_squad': 'activities.arnis_team',
    'activities.arnis_corps_squad_(2c)': 'activities.arnis_team',
    'activities.arnis_corps_squad_(3c)': 'activities.arnis_team',
    'activities.arnis_exhibition_team': 'activities.arnis_exhibition_team',
    'activities.arts_guild': 'activities.arts_guild',
    'activities.athletic_council': 'activities.athletic_council',
    'activities.bac_25_pc': 'activities.bac_25_pc',
    'activities.badminton_club': 'activities.badminton_club',
    'activities.badminton_corps_squad': 'activities.badminton_team',
    'activities.baguio_youth_marathon_(4c)': 'activities.baguio_youth_marathon',
    'activities.basketball_club': 'activities.basketball_club',
    'activities.basketball_corps_squad': 'activities.basketball_team',
    'activities.basketball_corps_squad_(2c)': 'activities.basketball_team',
    'activities.basketball_intramurals_1980': 'activities.basketball_team',
    'activities.board_of_directors,_dialectic_society': 'activities.dialectic_society',
    'activities.board_of_governors': 'activities.board_of_governors',
    'activities.body_building_club': 'activities.body_building_club',
    'activities.boodles_committee,_dialectic_society': 'activities.dialectic_society',
    'activities.bowling_club': 'activities.bowling_club',
    'activities.boxing_club': 'activities.boxing_club',
    'activities.boxing_club_(3c)': 'activities.boxing_club',
    'activities.bridge_club': 'activities.bridge_club',
    'activities.cadet_announcing_and_briefing_squad': 'activities.cadet_squad',
    'activities.cadet_choir': 'activities.cadet_choir',
    'activities.cadet_combo': 'activities.cadet_combo',
    'activities.cadet_glee_club': 'activities.cadet_glee_club',
    'activities.cadet_speakers_club': 'activities.cadet_speakers_club',
    'activities.camera_club': 'activities.camera_club',
    'activities.camera_club_(2c)': 'activities.camera_club',
    'activities.catholic_action_group': 'activities.catholic_action_group',
    'activities.chess_club': 'activities.chess_club',
    'activities.chess_corps_squad': 'activities.chess_team',
    'activities.class_1980_(4c)': 'activities.class_of_1980',
    'activities.class_of_1980': 'activities.class_of_1980',
    'activities.class_of_1980_(1c)': 'activities.class_of_1980',
    'activities.class_of_1980_(2c)': 'activities.class_of_1980',
    'activities.class_of_1980_(3c)': 'activities.class_of_1980',
    'activities.class_of_1980_(3c).1': 'activities.class_of_1980',
    'activities.class_of_1980_(4c)': 'activities.class_of_1980',
    'activities.corps_board_and_staff': 'activities.corps_board_and_staff',
    'activities.debating_club': 'activities.debating_club',
    'activities.debating_council': 'activities.debating_council',
    'activities.delta_cross-country_team': 'activities.cross_country_team',
    'activities.dialectic_council': 'activities.dialectic_council',
    'activities.dialectic_society': 'activities.dialectic_society',
    'activities.editorial_board,_the_corps_75th_treasury': 'activities.editorial_board',
    'activities.equipment_committee': 'activities.equipment_committee',
    'activities.fencing_club': 'activities.fencing_club',
    'activities.fencing_club_(3c)': 'activities.fencing_club',
    'activities.fencing_club_(4c)': 'activities.fencing_club',
    'activities.fencing_corps_squad': 'activities.fencing_team',
    'activities.fencing_intramurals': 'activities.fencing_team',
    'activities.firstclass_policy_board': 'activities.firstclass_policy_board',
    'activities.glee_club': 'activities.glee_club',
    'activities.golf_club': 'activities.golf_club',
    'activities.gymastics_club': 'activities.gymnastics_club',
    'activities.gymnastics_club': 'activities.gymnastics_club',
    'activities.gymnastics_corps_squad': 'activities.gymnastics_team',
    'activities.gymnastics_society': 'activities.gymnastics_club',
    'activities.gynamstics_corps_squad': 'activities.gymnastics_team',
    'activities.holy_name_society': 'activities.holy_name_society',
    'activities.honor_committee': 'activities.honor_committee',
    'activities.hop_committee': 'activities.hop_committee',
    'activities.judo_club': 'activities.judo_club',
    'activities.judo_corps_squad': 'activities.judo_team',
    'activities.junior_track_and_field_corps_squad': 'activities.track_and_field_team',
    'activities.karate_club': 'activities.karate_club',
    'activities.karate_corps_squad': 'activities.karate_team',
    'activities.karate_intramurals': 'activities.karate_team',
    'activities.lawn_tennis_club': 'activities.lawn_tennis_club',
    'activities.lawn_tennis_corps_squad': 'activities.lawn_tennis_team',
    'activities.mess_council': 'activities.mess_council',
    'activities.n/a': 'activities.n_a',
    'activities.national_archery_association_of_the_philippines': 'activities.archery_association',
    'activities.peacemaker': 'activities.peacemaker',
    'activities.peacemaker_board_and_staff': 'activities.peacemaker_board_and_staff',
    'activities.peemayer_board_and_staff': 'activities.peacemaker_board_and_staff',
    'activities.pelota_club': 'activities.pelota_club',
    'activities.pelota,_intramurals': 'activities.pelota_team',
    'activities.protestant_group': 'activities.protestant_group',
    'activities.radio_club': 'activities.radio_club',
    'activities.radio_club_(2c)': 'activities.radio_club',
    'activities.radio_club_(3c)': 'activities.radio_club',
    'activities.ring_committee': 'activities.ring_committee',
    'activities.scuba_diving_club': 'activities.scuba_diving_club',
    'activities.secretary': 'activities.secretary',
    "activities.ship_for_southeast_asian_youth_program_'78": 'activities.southeast_asian_youth_program',
    'activities.silent_drill_company': 'activities.silent_drill_company',
    'activities.skin_diving_club': 'activities.skin_diving_club',
    'activities.soccer_club': 'activities.soccer_club',
    'activities.soccer_corps_squad': 'activities.soccer_team',
    'activities.soccer_corps_squad_(4c)': 'activities.soccer_team',
    'activities.soccer_youth_team': 'activities.soccer_team',
    'activities.soccer_youth_team_(2c)': 'activities.soccer_team',
    'activities.soccer_youth_team_(4c)': 'activities.soccer_team',
    'activities.soccer_youth_team_(4c).1': 'activities.soccer_team',
    'activities.softball_club': 'activities.softball_club',
    'activities.softball_corps_squad': 'activities.softball_team',
    'activities.softball_and_baseball_club': 'activities.softball_club',
    'activities.softball/baseball_club': 'activities.softball_club',
    'activities.suba_diving_club': 'activities.scuba_diving_club',
    'activities.swimming_club': 'activities.swimming_club',
    'activities.swimming_corps_squad': 'activities.swimming_team',
    'activities.sword_1980': 'activities.sword_1980',
    'activities.sword_board_and_staff': 'activities.sword_board_and_staff',
    'activities.sword_board_and_staff_(2c)': 'activities.sword_board_and_staff',
    'activities.table_tennis_club': 'activities.table_tennis_club',
    'activities.table_tennis_team': 'activities.table_tennis_team',
    'activities.technical_committee': 'activities.technical_committee',
    'activities.tennis_club': 'activities.tennis_club',
    'activities.the_corps_75th_treasury': 'activities.the_corps_75th_treasury',
    'activities.the_dialectic_society': 'activities.dialectic_society',
    'activities.track_and_field_club': 'activities.track_and_field_club',
    'activities.track_and_field_corps_squad': 'activities.track_and_field_team',
    'activities.track_and_field_corps_squad_(3c)': 'activities.track_and_field_team',
    'activities.upper_committee': 'activities.upper_committee',
    'activities.volleyball_club': 'activities.volleyball_club',
    'activities.volleyball_corps': 'activities.volleyball_team',
    'activities.volleyball_corps_squad': 'activities.volleyball_team',
    'activities.volleyball_intramurals': 'activities.volleyball_team',
    'activities.wrestling_club': 'activities.wrestling_club',
}

df = pd.read_csv('processed_dat/net_dat/net_roles_num.csv')
df.rename(columns=activity_map, inplace=True)
df = df.groupby(df.columns, axis=1).sum()  # merge columns with the same canonical name
df.to_csv('processed_dat/net_dat/net_roles_numclean.csv', index=False)
print(f"Saved: net_roles_numclean.csv  ({df.shape[1] - 1} canonical activity columns)")

### 2.3 Build Company Matrix

Extract company affiliation and encode as binary dummy variables (0 = not a member, 1 = member). Company-level leadership roles carried in the activities matrix are later merged here by summing — allowing the final weight to represent both membership and level of involvement.

In [ ]:
df = pd.read_csv('processed_dat/net_dat/total_net_clean.csv')

# Fix known transcription error: 'Foxrot' → 'Foxtrot'
df['company'] = df['company'].replace('Foxrot', 'Foxtrot')

# One-hot encode company membership
dummies = pd.get_dummies(df['company'])
df_comps = pd.concat([df[['id']], dummies], axis=1)

df_comps.to_csv('processed_dat/net_dat/net_comps.csv', index=False)
print(f"Saved: net_comps.csv  (companies: {list(dummies.columns)})") 

### 2.4 Merge Roles and Company Matrices

Company-level activity columns (e.g., `activities.a_company`) are remapped to company names then merged with the dummy-encoded company matrix by summing. The result is a single weighted affiliation matrix where higher values indicate greater involvement.

In [ ]:
company_map = {
    "activities.a_company": "Alfa",
    "activities.b_company": "Bravo",
    "activities.c_company": "Charlie",
    "activities.d_company": "Delta",
    "activities.e_company": "Echo",
    "activities.f_company": "Foxtrot",
    "activities.g_company": "Golf",
    "activities.h_company": "Hawk",
}

df_roles = pd.read_csv('processed_dat/net_dat/net_roles_numclean.csv')
df_roles.rename(columns=company_map, inplace=True)

df_comps = pd.read_csv('processed_dat/net_dat/net_comps.csv')

# Stack and sum rows sharing the same id to merge overlapping columns
merged = pd.concat([df_roles, df_comps], axis=0, ignore_index=True)
merged = merged.groupby('id', as_index=False).sum()

merged.to_csv('processed_dat/net_dat/rolescomps_num.csv', index=False)
print(f"Saved: rolescomps_num.csv  ({merged.shape[0]} cadets, {merged.shape[1] - 1} features)")